In [ ]:
import numpy as np
from manim import *

import icm_anim as anim

In [ ]:
class MarkovGeneration(Scene):
    """Generating from a Markov chain with three states, C, G, and A. The first state
    is drawn uniformly at random. Then the transitions out of the current state light
    up, one of them is drawn at random, and the new state joins the running sequence,
    along with its probability. This time, the chain generates the first phrase of
    Twinkle, Twinkle, Little Star."""

    def construct(self):
        R = 0.6
        FAINT = 0.25                                     # the opacity of idle transitions
        pos = {"C": np.array([0.0, 1.2, 0.0]), "G": np.array([-2.6, -2.5, 0.0]),
               "A": np.array([2.6, -2.5, 0.0])}
        P = {"C": {"C": "1/2", "G": "1/2", "A": "0"},
             "G": {"C": "0", "G": "1/2", "A": "1/2"},
             "A": {"C": "0", "G": "1/2", "A": "1/2"}}
        TEX = {"1/2": r"\tfrac{1}{2}", "1/3": r"\tfrac{1}{3}", "0": "0"}
        LOOP_ANGLE = {"C": 90, "G": 215, "A": -35}

        circles = {s: Circle(radius=R, color=anim.BLUE, stroke_width=4).move_to(p)
                   for s, p in pos.items()}
        names = {s: Text(s, font_size=46).move_to(p) for s, p in pos.items()}

        # every transition: an arrow (a loop for a state to itself) and its probability
        arrows, labels = {}, {}
        for a in pos:
            for b in pos:
                if a == b:
                    t = np.deg2rad(LOOP_ANGLE[a])
                    start = pos[a] + R * np.array([np.cos(t + 0.55), np.sin(t + 0.55), 0])
                    end = pos[a] + R * np.array([np.cos(t - 0.55), np.sin(t - 0.55), 0])
                    arrow = CurvedArrow(start, end, angle=-4.2, color=anim.IRON,
                                        stroke_width=3, tip_length=0.18)
                    spot = pos[a] + (R + 0.85) * np.array([np.cos(t), np.sin(t), 0])
                else:
                    d = (pos[b] - pos[a]) / np.linalg.norm(pos[b] - pos[a])
                    right = np.array([d[1], -d[0], 0.0])
                    start = pos[a] + d * (R + 0.08) + right * 0.17
                    end = pos[b] - d * (R + 0.08) + right * 0.17
                    arrow = Arrow(start, end, buff=0, color=anim.IRON, stroke_width=3,
                                  max_tip_length_to_length_ratio=0.07)
                    spot = (start + end) / 2 + right * 0.42
                arrows[a, b] = arrow.set_opacity(FAINT).set_fill(opacity=0, family=False)
                labels[a, b] = MathTex(TEX[P[a][b]]).scale(0.75).move_to(spot)

        heading = Text("Generated:", font_size=30).move_to([-5.2, 3.3, 0])
        self.play(*[Create(c) for c in circles.values()], *[FadeIn(n) for n in names.values()],
                  *[Create(arrow) for arrow in arrows.values()], FadeIn(heading), run_time=1.2)

        # the first state, uniformly at random
        origin = np.array([-6.0, -0.5, 0.0])
        starts = {s: Arrow(origin, pos[s] + (origin - pos[s]) / np.linalg.norm(origin - pos[s]) * (R + 0.08),
                           buff=0, color=anim.IRON, stroke_width=3,
                           max_tip_length_to_length_ratio=0.06) for s in pos}
        start_labels = {s: MathTex(TEX["1/3"]).scale(0.75).move_to(
                        starts[s].point_from_proportion(0.45) + np.array([0, 0.3, 0])) for s in pos}
        dot = Dot(origin, radius=0.1, color=anim.IRON)
        self.play(FadeIn(dot), *[GrowArrow(arrow) for arrow in starts.values()],
                  *[FadeIn(label) for label in start_labels.values()], run_time=0.8)
        self.play(starts["C"].animate.set_color(anim.RED), start_labels["C"].animate.set_color(anim.RED),
                  run_time=0.4)
        self.play(circles["C"].animate.set_fill(anim.GOLD, opacity=0.35), run_time=0.4)
        entry = MathTex(r"\text{C}\left(\tfrac{1}{3}\right)").scale(0.8)
        sequence = VGroup(entry).next_to(heading, RIGHT, buff=0.3)
        self.play(FadeIn(entry, shift=0.2 * UP), FadeOut(dot), *[FadeOut(arrow) for arrow in starts.values()],
                  *[FadeOut(label) for label in start_labels.values()], run_time=0.6)

        # each next state, drawn from the transitions out of the current one
        path = ["C", "C", "G", "G", "A", "A", "G"]
        for a, b in zip(path[:-1], path[1:]):
            out = [(a, c) for c in pos]
            self.play(*[arrows[e].animate.set_opacity(1.0).set_fill(opacity=0, family=False)
                        for e in out],
                      *[FadeIn(labels[e]) for e in out], run_time=0.45)
            self.play(arrows[a, b].animate.set_color(anim.RED).set_fill(opacity=0, family=False),
                      Indicate(labels[a, b], color=anim.RED, scale_factor=1.3), run_time=0.45)
            moves = [circles[a].animate.set_fill(opacity=0), circles[b].animate.set_fill(anim.GOLD, opacity=0.35)]
            if a == b:
                moves = [Indicate(circles[b], color=anim.GOLD, scale_factor=1.12)]
            entry = MathTex(rf"\text{{{b}}}\left({TEX[P[a][b]]}\right)").scale(0.8)
            entry.next_to(sequence, RIGHT, buff=0.3)
            sequence.add(entry)
            self.play(*moves, FadeIn(entry, shift=0.2 * UP), run_time=0.5)
            self.play(*[arrows[e].animate.set_color(anim.IRON).set_opacity(FAINT)
                        .set_fill(opacity=0, family=False) for e in out],
                      *[FadeOut(labels[e]) for e in out], run_time=0.3)

        total = MathTex(r"\tfrac{1}{3} \cdot \left(\tfrac{1}{2}\right)^6 = \tfrac{1}{192}").scale(0.85)
        total.next_to(sequence, DOWN, buff=0.35).align_to(sequence, LEFT)
        self.play(FadeIn(total), run_time=0.6)
        self.wait(2.0)
        # reset, so the loop restarts cleanly
        self.play(FadeOut(sequence), FadeOut(total), circles["G"].animate.set_fill(opacity=0),
                  run_time=0.6)
        self.wait(0.3)


anim.show(MarkovGeneration)